In [1]:
import numpy as np
import pandas as pd

In [2]:
dataset = pd.read_csv('../../Data/credit_data.csv')

In [3]:
dataset.head()

,Time,V1,V2,V3,V4,V5,V6,V7,V8,V9,...,V21,V22,V23,V24,V25,V26,V27,V28,Amount,Class
0,0.0,-1.359807,-0.072781,2.536347,1.378155,-0.338321,0.462388,0.239599,0.098698,0.363787,...,-0.018307,0.277838,-0.110474,0.066928,0.128539,-0.189115,0.133558,-0.021053,149.62,0
1,0.0,1.191857,0.266151,0.166480,0.448154,0.060018,-0.082361,-0.078803,0.085102,-0.255425,...,-0.225775,-0.638672,0.101288,-0.339846,0.167170,0.125895,-0.008983,0.014724,2.69,0
2,1.0,-1.358354,-1.340163,1.773209,0.379780,-0.503198,1.800499,0.791461,0.247676,-1.514654,...,0.247998,0.771679,0.909412,-0.689281,-0.327642,-0.139097,-0.055353,-0.059752,378.66,0
3,1.0,-0.966272,-0.185226,1.792993,-0.863291,-0.010309,1.247203,0.237609,0.377436,-1.387024,...,-0.108300,0.005274,-0.190321,-1.175575,0.647376,-0.221929,0.062723,0.061458,123.50,0
4,2.0,-1.158233,0.877737,1.548718,0.403034,-0.407193,0.095921,0.592941,-0.270533,0.817739,...,-0.009431,0.798278,-0.137458,0.141267,-0.206010,0.502292,0.219422,0.215153,69.99,0


In [7]:
dataset.shape

(284807, 31)

In [5]:
dataset['Class'].value_counts()

# 0 ---> legit transaction
# 1 ---> fraud transaction
# highly imbalance dataset

Class
0    284315
1       492
Name: count, dtype: int64

In [10]:
# seperating bata based on label
# legit = dataset[dataset.Class == 0]
# fraud = dataset[dataset.Class == 1]

groups = dict(tuple(dataset.groupby("Class")))

legit = groups[0]
fraud = groups[1]

In [12]:
legit.shape

(284315, 31)

In [13]:
fraud.shape

(492, 31)

### Undersampling

In [14]:
legit = legit.sample(n=500)

In [15]:
legit.shape

(500, 31)

## Concatenate the two dataframes

In pandas, `axis` defines the **direction of concatenation**:

* **`axis=0` → vertical concatenation (row-wise)**

  * Stacks DataFrames on top of each other
  * Number of **columns stays same**
  * Number of **rows increases**
  * Used when combining datasets with same structure

  Example result:

  ```
  (992, 31)
  ```

---

* **`axis=1` → horizontal concatenation (column-wise)**

  * Places DataFrames side by side
  * Number of **rows determined by index alignment**
  * Number of **columns increases**
  * If indexes don’t match, pandas fills missing rows with `NaN`

  Example result:

  ```
  (992, 62)
  ```

---

### Why row count stayed 992 in both cases?

Because `legit` and `fraud` have **different index values**.

When using `axis=1`, pandas takes the **union of indexes**, which equals:

[
\text{rows} = \text{len(legit)} + \text{len(fraud)}
]

So row count remains 992, but columns double (31 + 31 = 62).

---

### Rule to Remember

* Increase rows → `axis=0`
* Increase columns → `axis=1`
* `axis=1` aligns by index

For dataset rebalancing or stacking samples, always use:

```python
pd.concat([legit, fraud], axis=0, ignore_index=True)
```

### 📌 `ignore_index=True` — Short Note

When using `pd.concat()`:

By default, pandas **keeps the original index values** from the input DataFrames.

If those indexes overlap or are non-sequential, the resulting DataFrame may have:

* Duplicate index values
* Non-continuous indexing
* Confusing row labels

---

### What `ignore_index=True` Does

It **discards the original indexes** and creates a fresh, continuous index:

[
0, 1, 2, 3, \dots, n-1
]

So:

```python
pd.concat([legit, fraud], axis=0, ignore_index=True)
```

means:

* Stack rows vertically
* Reset index from scratch

---

### Why It’s Important

After filtering or sampling, your indexes look like:

```
130261
98136
136181
...
```

If you concatenate without resetting:

* Those large index numbers remain
* You may get duplicate indexes

With `ignore_index=True`:

```
0
1
2
3
...
```

Clean. Predictable. Safe for modeling.

---

### Rule

Use `ignore_index=True` when:

* Combining subsets of the same dataset
* Building a new dataset for ML
* You don’t care about preserving original row labels

Avoid it only when the index itself carries meaning (e.g., timestamps, IDs).


In [36]:
new_dataset = pd.concat([legit,fraud],axis=0,ignore_index=True)

In [33]:
new_dataset = pd.concat([legit,fraud],axis=1,ignore_index=True)

In [30]:
new_dataset.head()

,0,1,2,3,4,5,6,7,8,9,...,52,53,54,55,56,57,58,59,60,61
130261,79319.0,1.308952,0.252661,-0.236678,0.265601,0.315349,0.009461,-0.063024,0.043365,-0.097378,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
98136,66535.0,1.200111,-0.900132,1.024904,-0.710115,-1.443730,-0.054955,-1.177034,0.219435,-0.619349,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
136181,81583.0,-3.780972,2.849177,0.365728,-0.378113,-1.151602,-1.341871,0.316030,0.117071,1.531396,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
274055,165833.0,-0.894154,0.841774,0.398966,-0.541229,0.274361,-1.101565,0.869569,0.187810,-0.532452,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
120893,75997.0,0.337884,-1.870565,0.462921,0.546279,-1.508608,-0.020059,-0.062965,-0.026502,1.220547,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [34]:
new_dataset.shape

(992, 62)

In [37]:
new_dataset['Class'].value_counts()

Class
0    500
1    492
Name: count, dtype: int64